# 长度最小的子数组
## 问题
给定一个含有 n 个正整数的数组 nums 和一个正整数 target，找出数组中满足元素之和大于等于 target 的长度最小的连续子数组，并返回其长度。如果不存在符合条件的子数组，返回 0

[力扣题目链接](https://leetcode.cn/problems/minimum-size-subarray-sum/)

注意：子数组必须连续，元素之和只需要大于等于 target，不要求恰好等于 target

示例 1:
- 输入：target = 7, nums = [2,3,1,2,4,3]
- 输出：2
- 解释：子数组 [4,3] 的和为 7，长度为 2，是满足条件的最短子数组

示例 2:
- 输入：target = 4, nums = [1,4,4]
- 输出：1
- 解释：单个元素 [4] 就能满足条件

示例 3:
- 输入：target = 11, nums = [1,1,1,1,1,1,1,1]
- 输出：0
- 解释：整个数组的和仍然小于 target，不存在符合条件的子数组

## 思路
最直接的方法是枚举子数组的起点，再从这个起点向右累加，找到满足条件的最短子数组

由于 nums 中的元素都是正整数，右端向右移动时，子数组的和会增大；左端向右移动时，子数组的和会减小。利用这一特点，可以用滑动窗口维护一个连续区间，避免重复计算

滑动窗口也是双指针法的一种应用。和上一题从数组两端向中间移动不同，本题的两个指针都只向右移动

## 解法
### 暴力求解
用外层循环枚举起点 left，内层循环从 left 开始向右累加元素

一旦元素之和大于等于 target，就记录当前长度，并结束内层循环。因为固定起点后，继续向右扩展只会让子数组更长，不可能得到更短的结果

用 len(nums) + 1 初始化 result，表示还没有找到符合条件的子数组；遍历结束后，如果 result 没有更新，就返回 0


In [ ]:
class solution:
    def minSubArrayLen(self, target: int, nums: list[int]) -> int:
        result = len(nums) + 1

        for left in range(len(nums)):
            window_sum = 0

            for right in range(left, len(nums)):
                window_sum += nums[right]

                if window_sum >= target:
                    result = min(result, right - left + 1)
                    break

        return result if result <= len(nums) else 0


时间复杂度：$O(n^2)$ 最坏情况下没有子数组满足条件，需要枚举所有起点和终点，共进行 $n + (n - 1) + \cdots + 1$ 次累加

空间复杂度：$O(1)$ 只使用几个变量，通过累加维护子数组的和，不创建额外数组


### 滑动窗口
定义两个指针，维护左闭右闭区间 [left, right]：

左指针 left：指向窗口的起点，初始值为 0

右指针 right：指向窗口的终点，通过 for 循环向右遍历数组

用 window_sum 保存窗口内元素的和，用 result 保存已经找到的最短长度

每次移动右指针后：
- 将 nums[right] 加入 window_sum，扩大窗口
- 当 window_sum >= target 时，先用 right - left + 1 更新最短长度
- 再减去 nums[left]，并让 left 向右移动一位，尝试缩小窗口
- 重复缩小窗口，直到 window_sum < target，再继续移动右指针

**这里必须使用 while，不能只使用 if**：一次扩大窗口后，可能需要连续移除多个左端元素，才能找到以当前 right 结尾的最短有效窗口

以 target = 7, nums = [2,3,1,2,4,3] 为例：

| right | 加入元素后的窗口 | 缩小窗口的过程 | 当前最短长度 |
| --- | --- | --- | --- |
| 0 | [2]，和为 2 | 不满足条件，继续扩大 | 尚未找到 |
| 1 | [2,3]，和为 5 | 不满足条件，继续扩大 | 尚未找到 |
| 2 | [2,3,1]，和为 6 | 不满足条件，继续扩大 | 尚未找到 |
| 3 | [2,3,1,2]，和为 8 | 记录长度 4；移除 2 后，和为 6 | 4 |
| 4 | [3,1,2,4]，和为 10 | 记录长度 4；移除 3 后，[1,2,4] 的和为 7，记录长度 3；再移除 1，和为 6 | 3 |
| 5 | [2,4,3]，和为 9 | 记录长度 3；移除 2 后，[4,3] 的和为 7，记录长度 2；再移除 4，和为 3 | 2 |

为什么左指针不需要回退？

每次移除左端元素前，当前窗口已经满足条件，并且已经记录了它的长度。之后如果保留同一个起点再向右扩展，得到的窗口只会更长，不会改善答案，所以可以放心将 left 向右移动

**正整数条件很关键**：窗口的和不足时，继续移除左端正整数只会让和更小，因此此时应该扩大窗口。如果数组包含负数，就不能直接套用这里的滑动窗口写法


In [ ]:
class solution:
    def minSubArrayLen(self, target: int, nums: list[int]) -> int:
        left = 0
        window_sum = 0
        result = len(nums) + 1

        for right in range(len(nums)):
            window_sum += nums[right]

            while window_sum >= target:
                result = min(result, right - left + 1)
                window_sum -= nums[left]
                left += 1

        return result if result <= len(nums) else 0


时间复杂度：$O(n)$ 虽然 for 内部有 while，但 right 和 left 都只向右移动，每个元素最多加入窗口一次、移出窗口一次，两个指针总共最多移动 2n 次

空间复杂度：$O(1)$ 只保存指针、窗口和以及最短长度，不需要实际截取子数组

### 易错点
- 判断条件是 window_sum >= target，不能写成 == target
- 窗口是左闭右闭区间 [left, right]，长度为 right - left + 1
- 先记录有效窗口的长度，再移除左端元素；移除后窗口可能已经不满足条件
- 用 while 连续缩小窗口，不能只缩小一次
- 没有找到有效窗口时返回 0，不能直接返回初始化的 result
